In [1]:
import tkinter as tk
from tkinter import ttk
from tkinter import messagebox
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import nltk
import plotly.express as px
from datetime import datetime

In [2]:
nltk.download('vader_lexicon')

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\ADMIN\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


True

In [3]:
play_store_df = pd.read_csv(r"C:\Users\ADMIN\OneDrive\Desktop\elevance skills project\Play Store Data.csv")

In [4]:
play_store_df.head()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19M,"10,000+",Free,0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7M,"5,000,000+",Free,0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25M,"50,000,000+",Free,0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8M,"100,000+",Free,0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up


In [5]:
reviews_df = pd.read_csv(r"C:\Users\ADMIN\OneDrive\Desktop\elevance skills project\User Reviews.csv")
reviews_df.head()

,App,Translated_Review,Sentiment,Sentiment_Polarity,Sentiment_Subjectivity
0,10 Best Foods for You,I like eat delicious food. That's I'm cooking ...,Positive,1.00,0.533333
1,10 Best Foods for You,This help eating healthy exercise regular basis,Positive,0.25,0.288462
2,10 Best Foods for You,NaN,NaN,NaN,NaN
3,10 Best Foods for You,Works great especially going grocery store,Positive,0.40,0.875000
4,10 Best Foods for You,Best idea us,Positive,1.00,0.300000


In [6]:
#data cleaning
play_store_df.isnull().sum()

App                  0
Category             0
Rating            1474
Reviews              0
Size                 0
Installs             0
Type                 1
Price                0
Content Rating       1
Genres               0
Last Updated         0
Current Ver          8
Android Ver          3
dtype: int64

In [7]:
reviews_df.isnull().sum()

App                           0
Translated_Review         26868
Sentiment                 26863
Sentiment_Polarity        26863
Sentiment_Subjectivity    26863
dtype: int64

In [9]:
play_store_df = play_store_df.drop_duplicates(subset="App")
play_store_df.shape

(9660, 13)

In [10]:
play_store_df.isnull().sum()

App                  0
Category             0
Rating            1463
Reviews              0
Size                 0
Installs             0
Type                 1
Price                0
Content Rating       1
Genres               0
Last Updated         0
Current Ver          8
Android Ver          3
dtype: int64

In [11]:
#data transformation
play_store_df["Rating"] = play_store_df["Rating"].fillna(
    play_store_df["Rating"].median()
)

In [12]:
play_store_df["Type"] = play_store_df["Type"].fillna(
    play_store_df["Type"].mode()[0]
)

In [14]:
play_store_df["Current Ver"] = play_store_df["Current Ver"].fillna(
    play_store_df["Current Ver"].mode()[0]
)
play_store_df["Android Ver"] = play_store_df["Android Ver"].fillna(
    play_store_df["Android Ver"].mode()[0]
)
play_store_df["Content Rating"] = play_store_df["Content Rating"].fillna(
    play_store_df["Content Rating"].mode()[0]
)
play_store_df.isnull().sum()

App               0
Category          0
Rating            0
Reviews           0
Size              0
Installs          0
Type              0
Price             0
Content Rating    0
Genres            0
Last Updated      0
Current Ver       0
Android Ver       0
dtype: int64

In [15]:
play_store_df["Installs_Num"] = (
    play_store_df["Installs"]
    .str.replace(",", "", regex=False)
    .str.replace("+", "", regex=False)
)

play_store_df["Installs_Num"] = pd.to_numeric(
    play_store_df["Installs_Num"],
    errors="coerce"
)
play_store_df[["Installs", "Installs_Num"]].head()

,Installs,Installs_Num
0,"10,000+",10000.0
1,"500,000+",500000.0
2,"5,000,000+",5000000.0
3,"50,000,000+",50000000.0
4,"100,000+",100000.0


In [16]:
play_store_df["Reviews"] = pd.to_numeric(
    play_store_df["Reviews"],
    errors="coerce"
)
play_store_df["Reviews"].dtype

dtype('float64')

In [18]:
play_store_df = play_store_df[
    play_store_df["Size"] != "Varies with device"
]
play_store_df["Size"].head()

0     19M
1     14M
2    8.7M
3     25M
4    2.8M
Name: Size, dtype: str

In [20]:
play_store_df["Size"] = play_store_df["Size"].replace(
    "M", "", regex=True
)

play_store_df["Size"] = play_store_df["Size"].replace(
    "k", "", regex=True
)
play_store_df["Size"] = pd.to_numeric(
    play_store_df["Size"],
    errors="coerce"
)
play_store_df["Size"].head()

0    19.0
1    14.0
2     8.7
3    25.0
4     2.8
Name: Size, dtype: float64

In [21]:
#merge the files
merged_df = pd.merge(
    play_store_df,
    reviews_df,
    on="App",
    how="inner"
)
merged_df.head()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver,Installs_Num,Translated_Review,Sentiment,Sentiment_Polarity,Sentiment_Subjectivity
0,Coloring book moana,ART_AND_DESIGN,3.9,967.0,14.0,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up,500000.0,A kid's excessive ads. The types ads allowed a...,Negative,-0.250,1.000000
1,Coloring book moana,ART_AND_DESIGN,3.9,967.0,14.0,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up,500000.0,It bad >:(,Negative,-0.725,0.833333
2,Coloring book moana,ART_AND_DESIGN,3.9,967.0,14.0,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up,500000.0,like,Neutral,0.000,0.000000
3,Coloring book moana,ART_AND_DESIGN,3.9,967.0,14.0,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up,500000.0,NaN,NaN,NaN,NaN
4,Coloring book moana,ART_AND_DESIGN,3.9,967.0,14.0,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up,500000.0,I love colors inspyering,Positive,0.500,0.600000


In [53]:
#filtering
filtered_df = merged_df[
    merged_df["Rating"] >= 4.2
]
filtered_df = filtered_df[
    ~filtered_df["App"].str.contains(r"\d", regex=True, na=False)
]
filtered_df = filtered_df[
    filtered_df["Category"].str.startswith(("T", "P"), na=False)
]
filtered_df = filtered_df[
    filtered_df["Reviews"] > 1000
]
filtered_df.shape

(4398, 18)

In [55]:
filtered_df = filtered_df[
    (filtered_df["Size"] >= 20) &
    (filtered_df["Size"] <= 80)
]
filtered_df.head(10)

(1160, 18)

In [62]:
filtered_df = filtered_df.drop_duplicates(subset="App")
filtered_df.shape

(24, 18)

In [63]:
filtered_df["Category"].value_counts()

Category
PHOTOGRAPHY         6
TRAVEL_AND_LOCAL    6
PERSONALIZATION     5
PRODUCTIVITY        4
PARENTING           2
TOOLS               1
Name: count, dtype: int64

In [66]:
translation_dict = {
    "TRAVEL_AND_LOCAL": "Voyage et Local",
    "PRODUCTIVITY": "Productividad",
    "PHOTOGRAPHY": "写真"
}

In [67]:
filtered_df["Display_Category"] = (
    filtered_df["Category"].replace(translation_dict)
)

In [68]:
filtered_df[
    ["Category", "Display_Category"]
].drop_duplicates()

,Category,Display_Category
29396,PHOTOGRAPHY,写真
32276,TRAVEL_AND_LOCAL,Voyage et Local
34621,TOOLS,TOOLS
34901,PERSONALIZATION,PERSONALIZATION
36221,PRODUCTIVITY,Productividad
37141,PARENTING,PARENTING


In [69]:
filtered_df.shape

(24, 19)